In [ ]:
import sys
#!pip install polars --target ./my_custom_packages
#sys.path.append("./my_custom_packages") #
custom_path = "/stor/home/he4249/orange/my_custom_packages" # lifetimes
sys.path.insert(0, custom_path)

import polars as pl

pl.Config.set_tbl_rows(50)
pl.Config(tbl_cols = 50)

In [2]:
loc_mai = pl.read_parquet("Localisation_client_mai26.parquet")
#display(loc_mai.head())

loc_juin = pl.read_parquet("Loc_client_du_20260601_au_20260630.parquet")
loc_juin = loc_juin.filter(~pl.col("msisdn").str.contains("+", literal = True))
# loc_juin.select(pl.col("msisdn").str.contains("+", literal = True).sum())
# 289
#display(loc_juin.head())

loc_juillet = pl.read_parquet("Loc_client_juillet/*.parquet")
loc_juillet = loc_juillet.with_columns(
    pl.col("lac_ci").mode().first().over("MS").alias("top_user_site"),
    pl.col("MS").alias("msisdn")
)
loc_juillet = loc_juillet.group_by("msisdn").agg(
    pl.col("top_user_site").first().alias("site_id"),
    (pl.col("lac_ci") == pl.col("top_user_site")).sum().cast(pl.Int64).alias("nb_jours"),
)
#display(loc_juillet.head())

display(loc_mai.shape)
display(loc_juin.shape)
display(loc_juillet.shape)

display(loc_mai["msisdn"].n_unique())
display(loc_juin["msisdn"].n_unique())
display(loc_juillet["msisdn"].n_unique())

loc_quarter = pl.concat([loc_mai, loc_juin, loc_juillet]).group_by(["msisdn", "site_id"]).agg(
    pl.col("nb_jours").sum().alias("tt_nbjours")
).group_by("msisdn").agg(
    pl.col("site_id").sort_by("tt_nbjours", descending = True).first().alias("site_id"),
    pl.col("tt_nbjours").max().alias("nb_jours")
)
display(loc_quarter.head())
display(loc_quarter.shape)
display(loc_quarter["msisdn"].n_unique())


(4014830, 3)

(4115083, 3)

(2350353, 3)

4014830

4115083

2350353

msisdn,site_id,nb_jours
str,str,i64
"""261326517269""","""1010-61782""",24
"""261375759801""","""4000-41752""",15
"""261328344823""","""2700-6914826""",41
"""261325484462""","""3040-33481""",40
"""261325754163""","""3050-33841""",45


(4559462, 3)

4559462

In [3]:
forfait_mai = pl.read_parquet("msisdn_du_20260501_au_20260531.parquet")
#display(forfait_mai.head())

forfait_juin = pl.read_parquet("msisdn_01062026_au_30062026.parquet")
#display(forfait_juin.head())

forfait_juillet = pl.read_parquet("mai_juillet_2026_2attempt.parquet")
forfait_juillet = forfait_juillet.with_columns(
    pl.col("msisdn").cast(pl.Int64),
    pl.col("date").dt.to_string()
)
# missing_rows = forfait_mai.join(forfait_juillet, on = ["msisdn", "date"], how = "anti")
# missing_rows.shape
# 5 millions missing

forfait_juillet = forfait_juillet.filter(pl.col("date").str.starts_with("2026-07")).drop("CA")
#display(forfait_juillet.head())


canal_mapping = {
    "ASTSK3": "IVR",
    "USSDR123": "USSD",
    "USSDR111": "USSD",
    "OMON": "USSD",
    "OMONAPP": "APP OM",
    "WEBCARE": "WEB",
    "MYORANGE": "APP OM",
    "MBC": "MBC",
    "MAXIT": "MAXIT",
    "MAXITOM": "MAXIT",
    "NOT FOUND": None
}

forfait_juin = forfait_juin.with_columns(
    pl.col("origin").replace(canal_mapping).alias("Group_canal")
).drop("origin")

forfait_juillet = forfait_juillet.with_columns(
    pl.col("canal").replace(canal_mapping).alias("Group_canal")
).drop("canal")



forfait_mapping = pl.read_excel("rf_offer_now 1.xlsx").select(
    pl.col("bndle_longname").alias("Nom du forfait").str.to_lowercase(),
    pl.col("Gamme_groupe"),
    pl.col("Prix new").cast(pl.Float64).alias("CA") 
)
forfait_mapping = pl.concat([
    forfait_mapping,
    forfait_mapping.with_columns(pl.col("Nom du forfait").str.replace(" new", ""))
]).unique(subset = ["Nom du forfait"])



forfait_mai = forfait_mai.select(
    pl.col("date"),
    pl.col("msisdn"),
    pl.col("Nom du forfait").str.to_lowercase(),
    pl.col("Gamme_groupe"),
    pl.col("Group_canal")
).join(
    forfait_mapping.select("Nom du forfait", "CA"), 
    on = "Nom du forfait", 
    how = "left"
).select(["date", "msisdn", "Gamme_groupe", "Nom du forfait", "Group_canal", "CA"])

forfait_juin = forfait_juin.select(
    pl.col("date").dt.date().dt.to_string(),
    (pl.lit("261") + pl.col("msisdn").cast(pl.String)).cast(pl.Int64).alias("msisdn"),
    pl.col("Nom du forfait").str.to_lowercase(),
    pl.col("Group_canal")
).join(
    forfait_mapping, 
    on = "Nom du forfait", 
    how = "left"
).select(["date", "msisdn", "Gamme_groupe", "Nom du forfait", "Group_canal", "CA"])

forfait_juillet = forfait_juillet.select(
    pl.col("date").str.strptime(pl.Date, "%Y-%m-%d").dt.to_string(),
    (pl.lit("261") + pl.col("msisdn").cast(pl.String)).cast(pl.Int64).alias("msisdn"),
    pl.col("Nom du forfait").str.to_lowercase(),
    pl.col("Group_canal")
).join(
    forfait_mapping, 
    on = "Nom du forfait", 
    how = "left"
).select(["date", "msisdn", "Gamme_groupe", "Nom du forfait", "Group_canal", "CA"])


forfait_quarter = pl.concat([forfait_mai, forfait_juin, forfait_juillet])


extra_prices = {
    "25 sms vers tout operateur|25 …": 500.0,
    "internet 30 000 ar": 30000.0,
    "lany crédit 1 000 ar": 1000.0,
    "forfait boost 20go": 35000.0,
    "voix orange 3h|voix orange 3h|…": 7500.0,
    "sera 3,25go": 10000.0,
    "appel & sms 100 000 ar": 100000.0,
    "lany crédit 500 ar": 500.0,
    "optinet 100": 150000.0,
    "bp om 2000": 2000.0,
    "forfait voix zone b 1h": 2500.0,
    "freefiber 100go": 169000.0,
    "be connect 5go 7 jours": 30000.0,
    "50 sms vers orange|50 sms vers…": 500.0,
    "optinet 50": 99000.0,
    "hello monde 15 ": 15000.0,
    "voix tout operateur 1h|voix to…": 4000.0,
    "rechargement": 1000.0,
    "sera 6": 18000.0,
    "freefiber 200go": 259000.0,
    "freefiber 50go 3 fois": 252000.0,
    "be connect 5go 30 jours": 30000.0,
    "pass 2 semaines": 10000.0,
    "forfait boost 5go": 10000.0,
    "lany crédit 3000 ar": 3000.0,
    "sera 2,2go": 7000.0,
    "freefiber 50go 6 fois": 474000.0,
    "sera 9": 27000.0,
    "pass 1 semaine": 5000.0,
    "paré 60mn": 3000.0,
    "optinet 500": 350000.0,
    "voix orange 1h|voix orange 1h|…": 2500.0,
    "sera 250 mo": 1000.0,
    "voix flotte 1h|voix flotte 1h|…": 1500.0,
    "aôonnaaa rahariva": 2000.0,
    "voix flotte 3h|voix flotte 3h|…": 4000.0,
    "hello monde 5 ": 5000.0,
    "voix tout operateur 10h|voix t…": 25000.0,
    "freefiber 50go": 89000.0,
    "appel & sms 40 000 ar": 40000.0,
    "paré 120mn": 5000.0,
    "optinet 2": 49000.0,
    "lany data akama plus": 1000.0,
    "lany data be 5000": 5000.0,
    "optinet 5": 69000.0,
    "lany crédit  500 ar": 500.0,
    "24000ar ttc|24000ar ttc|24000a…": 24000.0,
    "voix orange 10h|voix orange 10…": 15000.0,
    "optinet 15": 89000.0,
    "forfait voix zone c 1h": 3000.0,
    "voix inde 1h|voix inde 1h|1h i…": 5000.0,
    "hello monde 40 ": 40000.0,
    "freefiber 100go 3 mois": 492000.0,
    "internet 500 000 ar": 500000.0,
    "lany crédit 3 000 ar": 3000.0,
    "aôonnaaa maître": 3000.0,
    "aôonnaaa boss": 5000.0,
    "france tv 1000": 1000.0,
    "lany crédit 1000 ar": 1000.0,
    "be connect 1,2 go": 10000.0,
    "lany data akama plus": 1000.0,
    "sera 500 mo": 2000.0,
    "hello horizon 15 ": 15000.0,
    "lany data be connect 550mo": 1000.0,
    "home confort": 75000.0,
    "hello horizon 5 ": 5000.0,
    "veedz 5500": 5500.0,
    "sera 15": 45000.0,
    "voix tout operateur 5h|voix to…": 12500.0,
    "sera 3": 9000.0,
    "internet 300 000 ar": 300000.0,
    "internet 100 000 ar": 100000.0
}

forfait_quarter = forfait_quarter.with_columns(
    CA = pl.col("CA").fill_null(
        pl.col("Nom du forfait").replace(extra_prices)
    )
)


gamme_extra_mapping = {
    # Divertissement, streaming
    "france tv 1000": "Divertissement", "france tv 2000": "Divertissement", "france tv 8000": "Divertissement",
    "new world tv 1000": "Divertissement", "new world tv 2000": "Divertissement", "new world tv 8000": "Divertissement",
    "veedz 1200": "Divertissement", "veedz 5500": "Divertissement", "veedz 11500": "Divertissement",
    
    # Internet, data, Fixe
    "home confort": "Akama/Be Connect", "freefiber 50go": "Akama/Be Connect", "freefiber 100go": "Akama/Be Connect",
    "freefiber 200go": "Akama/Be Connect", "freefiber 50go 3 fois": "Akama/Be Connect", "freefiber 50go 6 fois": "Akama/Be Connect",
    "freefiber 100go 3 mois": "Akama/Be Connect", "be connect 1,2 go": "Akama/Be Connect", "be connect 5go 7 jours": "Akama/Be Connect",
    "be connect 5go 30 jours": "Akama/Be Connect", "lany data akama plus": "Akama/Be Connect", "lany data be 5000": "Akama/Be Connect",
    "lany data be connect 550mo": "Akama/Be Connect", "optinet 2": "Akama/Be Connect", "optinet 5": "Akama/Be Connect",
    "optinet 15": "Akama/Be Connect", "optinet 50": "Akama/Be Connect", "optinet 100": "Akama/Be Connect", "optinet 500": "Akama/Be Connect",
    "internet 30 000 ar": "Akama/Be Connect", "internet 100 000 ar": "Akama/Be Connect", "internet 300 000 ar": "Akama/Be Connect", "internet 500 000 ar": "Akama/Be Connect",
    "sera 2,2go": "Akama/Be Connect", "sera 3,25go": "Akama/Be Connect", "sera 3": "Akama/Be Connect", "sera 6": "Akama/Be Connect", 
    "sera 9": "Akama/Be Connect", "sera 15": "Akama/Be Connect", "sera 250 mo": "Akama/Be Connect", "sera 500 mo": "Akama/Be Connect",
    "forfait boost 5go": "Akama/Be Connect", "forfait boost 20go": "Akama/Be Connect",

    # International
    "hello monde 5 ": "International", "hello monde 15 ": "International", "hello monde 40 ": "International",
    "hello horizon 5 ": "International", "hello horizon 15 ": "International", "voix inde 1h|voix inde 1h|1h i…": "International",

    # Voice, SMS, recharges 
    "rechargement": "Be", "lany crédit 500 ar": "Be", "lany crédit  500 ar": "Be", "lany crédit 1 000 ar": "Be",
    "lany crédit 1000 ar": "Be", "lany crédit 3000 ar": "Be", "lany crédit 3 000 ar": "Be",
    "bp om 2000": "Be", "pass 1 semaine": "Be", "pass 2 semaines": "Be",
    "paré 60mn": "Be", "paré 120mn": "Be",
    "25 sms vers tout operateur|25 …": "Be", "50 sms vers orange|50 sms vers…": "Be",
    "voix orange 1h|voix orange 1h|…": "Be", "voix orange 3h|voix orange 3h|…": "Be", "voix orange 10h|voix orange 10…": "Be",
    "voix flotte 1h|voix flotte 1h|…": "Be", "voix flotte 3h|voix flotte 3h|…": "Be",
    "voix tout operateur 1h|voix to…": "Be", "voix tout operateur 5h|voix to…": "Be", "voix tout operateur 10h|voix t…": "Be",
    "forfait voix zone b 1h": "Be", "forfait voix zone c 1h": "Be",
    "appel & sms 40 000 ar": "Be", "appel & sms 100 000 ar": "Be", "24000ar ttc|24000ar ttc|24000a…": "Be"
}

forfait_quarter = forfait_quarter.with_columns(
    Gamme_groupe = pl.col("Gamme_groupe").fill_null(
        pl.col("Nom du forfait").replace(gamme_extra_mapping)
    )
)

display(forfait_mai.shape)
display(forfait_juin.shape)
display(forfait_juillet.shape)

display(forfait_quarter.head())
display(forfait_quarter.shape)


(14266666, 6)

(15187366, 6)

(10085215, 6)

date,msisdn,Gamme_groupe,Nom du forfait,Group_canal,CA
str,i64,str,str,str,str
"""2026-05-05""",261320236812,"""Akama/Be Connect""","""akama up""","""IVR""","""500.0"""
"""2026-05-25""",261320274396,"""Akama/Be Connect""","""akama plus""","""APP OM""","""1000.0"""
"""2026-05-07""",261320251199,"""Be""","""be 2000 new""","""IVR""","""2000.0"""
"""2026-05-26""",261320248283,"""Akama/Be Connect""","""akama full""","""IVR""","""2000.0"""
"""2026-05-06""",261320454752,"""Akama/Be Connect""","""akama up""","""USSD""","""500.0"""


(39539247, 6)

In [5]:
display(forfait_quarter["msisdn"].n_unique())

3089198

In [4]:
'''loc_quarter.write_parquet("loc_quarter.parquet")
forfait_quarter.write_parquet("forfait_quarter.parquet")'''

'loc_quarter.write_parquet("loc_quarter.parquet")\nforfait_quarter.write_parquet("forfait_quarter.parquet")'